# Run experiment on Finance datasets

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os

print([np.__version__, pd.__version__])
np.set_printoptions(precision=3, suppress=True)


from src.data_preprocessing import preprocess_data
from src.plotting import plot_heatmap
from src.causal_matrix_evaluation import evaluate_causal_matrices
from src.matrix_utils import read_matrices_from_csv, save_matrices, get_summary_matrix
from src.run_causal_discovery import run_varlingam, run_varlingam_bootstrap, run_pcmci, run_rcv_varlingam, run_rcv_pcmci, run_tcdf

['1.24.4', '2.0.3']


In [5]:
def run_experiments(file_list, methods=['varlingam', 'pcmci', 'rcv_varlingam', 'rcv_pcmci', 'tcdf']):
    """
    Run causal discovery experiments on Finance datasets.
    
    Args:
        file_list: List of file identifiers to process
        methods: List of methods to run (default includes all methods)
    """
    results = {method: [] for method in methods}

    for file_id in file_list:
        print(f"Running experiments for {file_id}...")
        
        # Load ground truth
        ground_truth_path = f'data/real/Finance/ground_truths/{file_id}_adj.csv'
        ground_truth_matrices = read_matrices_from_csv(ground_truth_path)
        
        if ground_truth_matrices is None:
            print(f"Skipping {file_id} due to missing ground truth")
            continue
            
        # Get summary ground truth matrix
        ground_truth_summary = get_summary_matrix(ground_truth_matrices)
        
        for method in methods:
            # Load data
            data_path = f'data/real/Finance/returns/{file_id}_returns.csv'
            data = pd.read_csv(data_path)
            columns = data.columns.tolist()
            
            # Remove timestamp column if present
            for time_col in ['Date', 'timestamp']:
                if time_col in columns:
                    data = data.drop([time_col], axis=1)
                    columns.remove(time_col)
            
            data = data.values

            # Preprocess data
            data = preprocess_data(data, columns)

            # Run causal discovery method
            start_time = time.time()
            
            if method == 'varlingam':
                adjacency_matrices = run_varlingam(data)
            elif method == 'varlingam_bootstrap':
                adjacency_matrices = run_varlingam_bootstrap(data)
            elif method == 'pcmci':
                adjacency_matrices = run_pcmci(data, columns)
            elif method == 'rcv_varlingam':
                adjacency_matrices = run_rcv_varlingam(data)
            elif method == 'rcv_pcmci':
                adjacency_matrices = run_rcv_pcmci(data)
            elif method == 'tcdf':
                adjacency_matrices = run_tcdf(data)
            else:
                raise ValueError(f"Unknown method: {method}")
                
            runtime = round(time.time() - start_time, 4)

            # Trim adjacency matrices if needed
            if len(adjacency_matrices) > len(ground_truth_matrices):
                adjacency_matrices = adjacency_matrices[:len(ground_truth_matrices)]

            # Get summary matrix from method results
            method_summary = get_summary_matrix(adjacency_matrices)

            # Save results
            save_dir = f'results/real/Finance/{file_id}'
            os.makedirs(save_dir, exist_ok=True)
            
            # Save full adjacency matrices
            matrices_path = f'{save_dir}/adj_matrices_{method}.csv'
            save_matrices(adjacency_matrices, matrices_path)
            
            # Save summary matrix
            summary_path = f'{save_dir}/sum_adj_matrix_{method}.csv'
            save_matrices([method_summary], summary_path)

            # Evaluate both full and summary matrices
            full_evaluation = evaluate_causal_matrices(ground_truth_matrices, adjacency_matrices)
            summary_evaluation = evaluate_causal_matrices([ground_truth_summary], [method_summary])

            # Store results
            result = {
                'file_id': file_id,
                'Full_SHD': full_evaluation['shd'],
                'Full_F1': full_evaluation['f1'],
                'Full_F1_sign': full_evaluation['f1_sign'],
                'Full_Frobenius': full_evaluation['fro'],
                'Summary_SHD': summary_evaluation['shd'],
                'Summary_F1': summary_evaluation['f1'],
                'Summary_F1_sign': summary_evaluation['f1_sign'],
                'Summary_Frobenius': summary_evaluation['fro'],
                'runtime': runtime
            }
            results[method].append(result)

    # Save results for each method
    for method in methods:
        df_results = pd.DataFrame(results[method])
        
        # Calculate statistics for numeric columns
        numeric_cols = df_results.select_dtypes(include=[np.number]).columns
        numeric_cols = [col for col in numeric_cols if col not in ['file_id']]
        avg_result = df_results[numeric_cols].mean()
        std_result = df_results[numeric_cols].std()

        # Create summary row
        summary = {
            'file_id': 'Overall Average',
            **{col: (f"{avg_result[col]:.4f} ± {std_result[col]:.4f}" 
                    if col != 'runtime' else f"{avg_result[col]:.4f}")
               for col in numeric_cols}
        }

        # Add summary to results
        df_results = pd.concat([df_results, pd.DataFrame([summary])], ignore_index=True)
        
        # Save to CSV
        output_path = f'results/real/Finance/performance_{method}.csv'
        df_results.to_csv(output_path, index=False)

In [7]:
# Define file list to process
files_to_process = [
    # "random-rels_20_1A",
    # "random-rels_20_1B",
    # "random-rels_20_1C",
    # "random-rels_20_1D",
    # "random-rels_20_1E",
    # "random-rels_20_1_3",
    # "random-rels_40_1_3",
    # "random-rels_40_1",
    "manyinputs"
]

# Define methods to run
methods_to_run = ['varlingam', 'pcmci', 'rcv_varlingam', 'rcv_pcmci', 'tcdf']

# Run experiments on specified files
run_experiments(files_to_process, methods=methods_to_run)

Running experiments for manyinputs...

##
## Step 1: PC1 algorithm for selecting lagged conditions
##

Parameters:
independence test = par_corr
tau_min = 1
tau_max = 3
pc_alpha = [0.05]
max_conds_dim = None
max_combinations = 1



## Resulting lagged parent (super)sets:

    Variable S0 has 2 link(s):
        (S20 -1): max_pval = 0.00000, |min_val| =  0.117
        (S14 -1): max_pval = 0.02366, |min_val| =  0.036

    Variable S1 has 3 link(s):
        (S20 -1): max_pval = 0.00000, |min_val| =  0.125
        (S0 -2): max_pval = 0.00007, |min_val| =  0.063
        (S14 -1): max_pval = 0.01840, |min_val| =  0.037

    Variable S2 has 4 link(s):
        (S20 -1): max_pval = 0.00003, |min_val| =  0.066
        (S14 -1): max_pval = 0.00464, |min_val| =  0.045
        (S10 -3): max_pval = 0.03269, |min_val| =  0.034
        (S2 -2): max_pval = 0.03623, |min_val| =  0.033

    Variable S3 has 4 link(s):
        (S10 -3): max_pval = 0.00005, |min_val| =  0.065
        (S21 -1): max_pval = 0.01